# Data Preprocessing
## Skin Health Analyzer 

**Dataset:** HAM10000 (Human Against Machine with 10000 training images)
- Download: https://www.kaggle.com/datasets/kmader/skin-lesion-analysis-toward-melanoma-detection

## Prep & Load Dataset

In [6]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import warnings

warnings.filterwarnings('ignore')

# ── Konfigurasi Direktori ──────────────────────────────────────────────
current_dir = Path(os.getcwd())
BASE_DIR = current_dir.parent if current_dir.name == 'notebooks' else current_dir

RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

# Buat folder processed jika belum ada
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Library dimuat dan direktori siap!")
print(f"📁 Folder Raw: {RAW_DIR}")

✅ Library dimuat dan direktori siap!
📁 Folder Raw: c:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw


## Memproses Dataset HAM10000

In [7]:
print("1️⃣ Memproses dataset HAM10000...")

ham_meta = pd.read_csv(RAW_DIR / 'HAM10000_metadata.csv')
img_dir1 = RAW_DIR / 'HAM10000_images_part_1'
img_dir2 = RAW_DIR / 'HAM10000_images_part_2'

# Mapping ID ke file fisik gambar
image_paths = {}
for folder in [img_dir1, img_dir2]:
    if folder.exists():
        for img_path in folder.glob('*.jpg'):
            image_paths[img_path.stem] = str(img_path.resolve())

ham_meta['image_path'] = ham_meta['image_id'].map(image_paths)
ham_meta = ham_meta.dropna(subset=['image_path'])

# Pilih kolom inti & isi data yang kosong (fillna)
df_ham = ham_meta[['image_path', 'dx', 'age', 'sex', 'localization']].copy()
df_ham['age'] = df_ham['age'].fillna(df_ham['age'].median())

print(f"✅ HAM10000 siap: {len(df_ham)} gambar terekstrak.")
df_ham.head(3)

1️⃣ Memproses dataset HAM10000...
✅ HAM10000 siap: 10015 gambar terekstrak.


,image_path,dx,age,sex,localization
0,C:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw\HA...,bkl,80.0,male,scalp
1,C:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw\HA...,bkl,80.0,male,scalp
2,C:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw\HA...,bkl,80.0,male,scalp


## Memproses Dataset SD101

In [8]:
print("2️⃣ Memproses dataset tambahan SD101...")

sd101_dir = RAW_DIR / 'SD101'
sd101_data = []
valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}

if sd101_dir.exists():
    for class_folder in sd101_dir.iterdir():
        if class_folder.is_dir():
            class_name = class_folder.name
            
            for img_path in class_folder.rglob('*'):
                if img_path.suffix.lower() in valid_extensions:
                    sd101_data.append({
                        'image_path': str(img_path.resolve()),
                        'dx': class_name,
                        'age': 0.0,              # Nilai dummy agar tabel selaras
                        'sex': 'unknown',        # Nilai dummy
                        'localization': 'face'   # Karena data SD101 spesifik di wajah
                    })
else:
    print("⚠️ Peringatan: Folder SD101 tidak ditemukan!")

df_sd101 = pd.DataFrame(sd101_data)
print(f"✅ SD101 siap: {len(df_sd101)} gambar terekstrak dari {df_sd101['dx'].nunique()} kelas.")
df_sd101.head(3)

2️⃣ Memproses dataset tambahan SD101...
✅ SD101 siap: 10360 gambar terekstrak dari 9 kelas.


,image_path,dx,age,sex,localization
0,C:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw\SD...,acne,0.0,unknown,face
1,C:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw\SD...,acne,0.0,unknown,face
2,C:\G\Bootcamp\PORTOFOLIO\DermaScan\data\raw\SD...,acne,0.0,unknown,face


## Penggabungan (Concatenation) & Pengecekan

In [9]:
print("3️⃣ Menggabungkan HAM10000 dan SD101...")

# Gabungkan baris dari kedua dataframe
df_combined = pd.concat([df_ham, df_sd101], ignore_index=True)

print(f"📊 Total baris dataset final: {len(df_combined)} gambar\\n")
print("Distribusi Kelas Keseluruhan:")
print(df_combined['dx'].value_counts())

3️⃣ Menggabungkan HAM10000 dan SD101...
📊 Total baris dataset final: 20375 gambar\n
Distribusi Kelas Keseluruhan:
dx
nv                   6705
SJS-TEN              3164
Nail_psoriasis       2520
Vitiligo             2016
acne                 1148
mel                  1113
bkl                  1099
hyperpigmentation     700
bcc                   514
akiec                 327
normal                263
dry                   257
oily                  229
vasc                  142
df                    115
combination            63
Name: count, dtype: int64


## Penyimpanan Final

In [10]:
print("4️⃣ Menyimpan dataset...")

CLEAN_PATH = PROCESSED_DIR / 'combined_clean.csv'
df_combined.to_csv(CLEAN_PATH, index=False)

print(f"🎉 Selesai! Dataset gabungan berhasil diamankan di:\\n👉 {CLEAN_PATH}")

4️⃣ Menyimpan dataset...
🎉 Selesai! Dataset gabungan berhasil diamankan di:\n👉 c:\G\Bootcamp\PORTOFOLIO\DermaScan\data\processed\combined_clean.csv
